# CogAttention — Stream Segregation (Cocktail Party)

**Track:** Attention — Sustained Attention
**Benchmark:** CogAttention v1.0
**Task:** stream_segregation

---

## Methodology

Tests stream segregation — tracking one interleaved conversation while ignoring a distractor stream, with optional breakthrough keyword detection. Based on Dichotic Listening / Cocktail Party effect (Cherry, 1953).

### Cognitive Science Grounding

- **Dichotic Listening / Cocktail Party Effect** (Cherry, 1953): attending to one conversation while ignoring another interleaved stream
- At harder levels, the model must also detect a breakthrough keyword in the ignored stream — testing divided attention under load

### Difficulty Scaling

Easy: 2 streams, 8 turns each | Medium: 2 streams, 15 turns | Hard: 2 streams, 20 turns + breakthrough | Expert: 2 streams, 30 turns + breakthrough | Frontier: 3 interleaved streams, 40 turns + breakthrough

### Scoring

Primary assertion: correct answer from target stream. Breakthrough assertion (when applicable): detect ALERT keyword in ignored stream.

All instances are procedurally generated from a seed with programmatic ground truth.
No static datasets. 7 layers of contamination resistance including canary strings,
zero lexical overlap (Semantic NIAH), and seed-based regeneration.

---

`<!-- COGATTENTION-BENCH-CANARY-2AF5EF538585 -->`


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 2: Imports + Inline Helpers
# CogAttention — Sustained Attention
# ══════════════════════════════════════════════════════════════════════

import kaggle_benchmarks as kbench

import json
import re

def extract_answer_block(response):
    for pat in [r"ANSWER:\s*(.*)", r"Answer:\s*(.*)", r"answer:\s*(.*)"]:
        match = re.search(pat, response, re.DOTALL | re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return response.strip()

def extract_numbered_answers(response):
    answer_block = extract_answer_block(response)
    results = {}
    matches = re.findall(
        r"(\d+)\s*[.):\-]\s*(.+?)(?=\n\d+\s*[.):\-]|\Z)",
        answer_block, re.DOTALL,
    )
    for num, val in matches:
        results[num] = val.strip().rstrip(".")
    return results

def extract_list_items(response):
    answer_block = extract_answer_block(response)
    bullets = re.findall(r"[-\u2022]\s*(.+?)(?:\n|$)", answer_block)
    if bullets:
        return [b.strip().rstrip(".") for b in bullets]
    numeric_items = re.findall(
        r'[\$]?\d{1,3}(?:,\d{3})*(?:\.\d+)?(?:\s*(?:\xb0[CF]|mg/L|%|\$))?',
        answer_block,
    )
    if numeric_items and len(numeric_items) >= 2:
        return [x.strip() for x in numeric_items]
    if "," in answer_block:
        items = [x.strip().rstrip(".") for x in answer_block.split(",")]
        return [x for x in items if x]
    lines = [l.strip().rstrip(".") for l in answer_block.split("\n") if l.strip()]
    return lines if lines else ([answer_block] if answer_block else [])

def extract_person_item_pairs(response):
    answer_block = extract_answer_block(response)
    results = {}
    for pat in [
        r"[-\u2022]?\s*(\w+)\s*:\s*(.+?)(?:\n|$)",
        r"[-\u2022]?\s*(\w+)\s+holds?\s+(?:a\s+)?(.+?)(?:\n|$)",
    ]:
        matches = re.findall(pat, answer_block, re.IGNORECASE)
        if matches:
            for name, item in matches:
                results[name.strip()] = item.strip().rstrip(".")
            break
    return results

def fuzzy_value_match(predicted, gold):
    pred_clean = re.sub(r"\s+", " ", predicted.strip().lower())
    gold_clean = re.sub(r"\s+", " ", gold.strip().lower())
    if pred_clean == gold_clean:
        return True
    if gold_clean in pred_clean:
        return True
    try:
        pred_num = float(re.sub(r"[,$%\xb0]", "", predicted))
        gold_num = float(re.sub(r"[,$%\xb0]", "", gold))
        return pred_num == gold_num
    except (ValueError, TypeError):
        pass
    return False

def _escape_for_regex(s):
    return re.escape(s).replace(r"\ ", r"\s+")


def run_assertions_stream_segregation(response, gold, kbench):
    gold_num = gold["first_number"]
    if gold_num and gold_num != "unknown":
        pattern = rf"\b{re.escape(gold_num)}\b"
        kbench.assertions.assert_contains_regex(
            pattern, response,
            expectation=f"First number in stream A should be '{gold_num}'"
        )
    if gold["has_breakthrough"]:
        kbench.assertions.assert_contains_regex(
            r"(?i)\byes\b", response,
            expectation="Should detect ALERT breakthrough in stream B"
        )


print("CogAttention helpers loaded")
print(f"Task types: ['stream_segregation']")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 3: Task Definitions + Embedded Dataset
# ══════════════════════════════════════════════════════════════════════


@kbench.task(name="cogattention_stream_segregation")
def cogattention_stream_segregation(llm, prompt: str, gold_json: str, task_id: str, difficulty: str):
    """CogAttention stream_segregation task."""
    response = llm.prompt(prompt)
    gold = json.loads(gold_json)
    run_assertions_stream_segregation(response, gold, kbench)


# ── Embedded dataset ──────────────────────────────────────────────────
DATASET = json.loads(r'''
[
 {
  "task_id": "stream_easy_000",
  "task_type": "stream_segregation",
  "difficulty": "Easy",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about cooking recipe).\nCompletely ignore conversation B.\n\n[A] Let the mixture simmer for 42 minutes.\n[B] The train from the airport takes about 6 minutes.\n[A] The total cooking time should be about 71 minutes.\n[B] Check out is at 11:00 — leave bags at reception.\n[A] The total cooking time should be about 71 minutes.\n[B] Book a hotel near the central park for the best location.\n[A] Dice the celery into small cubes.\n[B] Book a hotel near the central park for the best location.\n\nQuestion 1: Based ONLY on conversation A (about cooking recipe), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"42\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_easy_001",
  "task_type": "stream_segregation",
  "difficulty": "Easy",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversation B.\n\n[A] Space each plant at least 14 inches apart.\n[B] Market capitalization reached $396 billion.\n[A] Water thoroughly every 9 days during autumn.\n[B] The debt-to-equity ratio stands at 1.37.\n[A] Plant the tomato seeds 2 inches deep.\n[B] Revenue from the Asia-Pacific region grew 5%.\n[A] Plant the sunflower seeds 1 inches deep.\n[B] Net profit margin improved to 23.8%.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"14\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_easy_002",
  "task_type": "stream_segregation",
  "difficulty": "Easy",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about repair instructions).\nCompletely ignore conversation B.\n\n[A] Reattach the panel and tighten screws to 5 Nm.\n[B] Set up begins at 8:00 — the venue opens at 15:00.\n[A] Apply epoxy to both surfaces before joining.\n[B] The photographer charges $313 per hour.\n[A] Let the joint set for at least 23 hours.\n[B] Reserve 27 round tables with 6 chairs each.\n[A] Replace the worn bearing with the new one from the kit.\n[B] Parking is available for 72 vehicles.\n\nQuestion 1: Based ONLY on conversation A (about repair instructions), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"5\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_easy_003",
  "task_type": "stream_segregation",
  "difficulty": "Easy",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about repair instructions).\nCompletely ignore conversation B.\n\n[A] Reattach the panel and tighten screws to 16 Nm.\n[B] The venue holds up to 106 guests.\n[A] First, disconnect the power supply completely.\n[B] Flowers should arrive by 10:00 on the day.\n[A] Replace the worn filter with the new one from the kit.\n[B] Set up begins at 10:00 — the venue opens at 14:00.\n[A] First, disconnect the power supply completely.\n[B] The band can play from 19:00 to 23:00.\n\nQuestion 1: Based ONLY on conversation A (about repair instructions), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"16\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_easy_004",
  "task_type": "stream_segregation",
  "difficulty": "Easy",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about sports commentary).\nCompletely ignore conversation B.\n\n[A] The attendance tonight is 49,866 spectators.\n[B] The test results will be available in 14 business days.\n[A] The attendance tonight is 14,961 spectators.\n[B] Avoid alcohol for at least 9 days post-procedure.\n[A] The attendance tonight is 69,751 spectators.\n[B] Schedule a follow-up if symptoms persist beyond 3 days.\n[A] The score is 2-1 at the end of the first half.\n[B] The recommended daily water intake is 2.0 liters.\n\nQuestion 1: Based ONLY on conversation A (about sports commentary), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"49,866\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_easy_005",
  "task_type": "stream_segregation",
  "difficulty": "Easy",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about repair instructions).\nCompletely ignore conversation B.\n\n[A] Reattach the panel and tighten screws to 18 Nm.\n[B] Parking is available for 99 vehicles.\n[A] Let the joint set for at least 5 hours.\n[B] The photographer charges $337 per hour.\n[A] Reattach the panel and tighten screws to 19 Nm.\n[B] Set up begins at 10:00 — the venue opens at 14:00.\n[A] Apply silicone to both surfaces before joining.\n[B] Reserve 5 round tables with 10 chairs each.\n\nQuestion 1: Based ONLY on conversation A (about repair instructions), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"18\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_easy_006",
  "task_type": "stream_segregation",
  "difficulty": "Easy",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about sports commentary).\nCompletely ignore conversation B.\n\n[A] The attendance tonight is 32,347 spectators.\n[B] Avoid gluten for at least 12 days post-procedure.\n[A] The attendance tonight is 53,845 spectators.\n[B] The recommended daily water intake is 2.8 liters.\n[A] Substitution: Sigrid replaces Tariq.\n[B] Take 500mg of ibuprofen twice daily.\n[A] The score is 0-2 at the end of the third quarter.\n[B] Blood pressure reading was 111/86.\n\nQuestion 1: Based ONLY on conversation A (about sports commentary), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"32,347\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_easy_007",
  "task_type": "stream_segregation",
  "difficulty": "Easy",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about cooking recipe).\nCompletely ignore conversation B.\n\n[A] Let the mixture simmer for 18 minutes.\n[B] Pack an umbrella — the weather forecast shows cold winds.\n[A] Let the mixture simmer for 10 minutes.\n[B] The flight departs at 12:45 from terminal 3.\n[A] First, preheat the oven to 184 degrees.\n[B] Book a hotel near the cathedral for the best location.\n[A] Season with salt, pepper, and a pinch of turmeric.\n[B] The train from the airport takes about 41 minutes.\n\nQuestion 1: Based ONLY on conversation A (about cooking recipe), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"18\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_medium_008",
  "task_type": "stream_segregation",
  "difficulty": "Medium",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversation B.\n\n[A] The soil pH should be between 6.0 and 7.4.\n[B] The stock trades at a P/E ratio of 34.0.\n[A] Add potassium fertilizer once every 6 weeks.\n[B] The stock trades at a P/E ratio of 34.8.\n[A] Water thoroughly every 4 days during spring.\n[B] The quarterly revenue increased by 7% year-over-year.\n[A] Space each plant at least 17 inches apart.\n[B] Net profit margin improved to 8.8%.\n[A] The soil pH should be between 6.1 and 7.4.\n[B] Dividends per share will be $2.84.\n[A] Water thoroughly every 10 days during autumn.\n[B] Revenue from the North American region grew 16%.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"6.0\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_medium_009",
  "task_type": "stream_segregation",
  "difficulty": "Medium",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about sports commentary).\nCompletely ignore conversation B.\n\n[A] Injury time will be 5 minutes.\n[B] The follow-up appointment is in 7 weeks.\n[A] The referee issued a red card for the foul.\n[B] Schedule a follow-up if symptoms persist beyond 3 days.\n[A] Substitution: Haruto replaces Bram.\n[B] The follow-up appointment is in 7 weeks.\n[A] The corner kick is taken by Ines.\n[B] Exercise for at least 27 minutes daily.\n[A] The score is 3-1 at the end of the third quarter.\n[B] Limit sodium intake to 1759mg per day.\n[A] The match has been played in cold winds conditions.\n[B] Apply the moisturizing cream 2 times per day.\n\nQuestion 1: Based ONLY on conversation A (about sports commentary), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"5\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_medium_010",
  "task_type": "stream_segregation",
  "difficulty": "Medium",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversation B.\n\n[A] Expect germination in 7 to 13 days.\n[B] The quarterly revenue increased by 17% year-over-year.\n[A] Watch for caterpillars — treat with neem oil if spotted.\n[B] Net profit margin improved to 24.3%.\n[A] Space each plant at least 24 inches apart.\n[B] Operating costs are projected at $441 million.\n[A] Mulch with wood chips to retain moisture.\n[B] The debt-to-equity ratio stands at 1.84.\n[A] Plant the lettuce seeds 2 inches deep.\n[B] Operating costs are projected at $421 million.\n[A] Prune the sunflower back to 15 inches in March.\n[B] Net profit margin improved to 22.3%.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"7\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_medium_011",
  "task_type": "stream_segregation",
  "difficulty": "Medium",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about sports commentary).\nCompletely ignore conversation B.\n\n[A] Olena scored from 27 yards out.\n[B] Limit sodium intake to 2297mg per day.\n[A] Paloma scored from 35 yards out.\n[B] Avoid gluten for at least 4 days post-procedure.\n[A] Injury time will be 4 minutes.\n[B] Limit sodium intake to 2047mg per day.\n[A] The referee issued a red card for the foul.\n[B] Blood pressure reading was 136/82.\n[A] Injury time will be 4 minutes.\n[B] Blood pressure reading was 150/78.\n[A] The match has been played in sunshine conditions.\n[B] The follow-up appointment is in 2 weeks.\n\nQuestion 1: Based ONLY on conversation A (about sports commentary), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"27\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_medium_012",
  "task_type": "stream_segregation",
  "difficulty": "Medium",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversation B.\n\n[A] The soil pH should be between 5.9 and 6.5.\n[B] Revenue from the Asia-Pacific region grew 7%.\n[A] Plant the sunflower seeds 0.25 inches deep.\n[B] The debt-to-equity ratio stands at 2.14.\n[A] Mulch with wood chips to retain moisture.\n[B] Market capitalization reached $476 billion.\n[A] Prune the lettuce back to 13 inches in March.\n[B] Net profit margin improved to 5.5%.\n[A] Expect germination in 9 to 15 days.\n[B] Net profit margin improved to 23.1%.\n[A] Water thoroughly every 4 days during spring.\n[B] Capital expenditure is budgeted at $47 million.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"5.9\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_medium_013",
  "task_type": "stream_segregation",
  "difficulty": "Medium",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about sports commentary).\nCompletely ignore conversation B.\n\n[A] The score is 1-3 at the end of the first half.\n[B] Apply the moisturizing cream 2 times per day.\n[A] The score is 4-0 at the end of the first half.\n[B] Exercise for at least 12 minutes daily.\n[A] The corner kick is taken by Ravi.\n[B] Exercise for at least 39 minutes daily.\n[A] The referee issued a red card for the foul.\n[B] Take 500mg of amoxicillin twice daily.\n[A] The match has been played in sunshine conditions.\n[B] Blood pressure reading was 155/69.\n[A] Substitution: Zain replaces Sigrid.\n[B] The follow-up appointment is in 2 weeks.\n\nQuestion 1: Based ONLY on conversation A (about sports commentary), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"1\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_medium_014",
  "task_type": "stream_segregation",
  "difficulty": "Medium",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about cooking recipe).\nCompletely ignore conversation B.\n\n[A] Add 45 tablespoons of olive oil to the pan.\n[B] The guided tour starts at 10:00 near the main square.\n[A] Serve on a warm plate alongside rice.\n[B] The train from the airport takes about 39 minutes.\n[A] Stir occasionally until the sauce thickens.\n[B] The guided tour starts at 13:00 near the main square.\n[A] The total cooking time should be about 72 minutes.\n[B] Book a hotel near the central park for the best location.\n[A] Serve on a warm plate alongside bread.\n[B] The rental car pickup is at the east exit.\n[A] Garnish with fresh cilantro before serving.\n[B] The rental car pickup is at the main lobby.\n\nQuestion 1: Based ONLY on conversation A (about cooking recipe), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"45\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_medium_015",
  "task_type": "stream_segregation",
  "difficulty": "Medium",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversation B.\n\n[A] Prune the sunflower back to 18 inches in October.\n[B] The board approved a $57 million share buyback.\n[A] Space each plant at least 7 inches apart.\n[B] Capital expenditure is budgeted at $59 million.\n[A] Plant the lettuce seeds 0.5 inches deep.\n[B] Dividends per share will be $0.59.\n[A] Plant the sunflower seeds 0.25 inches deep.\n[B] The debt-to-equity ratio stands at 1.05.\n[A] Watch for caterpillars — treat with neem oil if spotted.\n[B] The stock trades at a P/E ratio of 26.3.\n[A] Expect germination in 7 to 15 days.\n[B] The board approved a $274 million share buyback.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"18\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_hard_016",
  "task_type": "stream_segregation",
  "difficulty": "Hard",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about repair instructions).\nCompletely ignore conversation B.\n\n[A] Use a 8mm wrench to loosen the bolt.\n[B] The photographer charges $216 per hour.\n[A] Use a 10mm wrench to loosen the bolt.\n[B] Reserve 29 round tables with 6 chairs each.\n[A] Apply contact cement to both surfaces before joining.\n[B] Parking is available for 88 vehicles.\n[A] Test the operation before restoring power.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** Catering quotes range from $47 to $76 per person.\n[A] Remove the 7 screws from the back panel.\n[B] Catering quotes range from $30 to $60 per person.\n[A] Apply epoxy to both surfaces before joining.\n[B] The cake needs to be ordered 4 days ahead.\n\nQuestion 1: Based ONLY on conversation A (about repair instructions), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"8\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_hard_017",
  "task_type": "stream_segregation",
  "difficulty": "Hard",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about sports commentary).\nCompletely ignore conversation B.\n\n[A] Nalini scored from 31 yards out.\n[B] Take 500mg of metformin twice daily.\n[A] Adaeze makes a save from close range.\n[B] Apply the hydrocortisone cream 3 times per day.\n[A] The score is 3-2 at the end of the second half.\n[B] Avoid caffeine for at least 8 days post-procedure.\n[A] The score is 1-2 at the end of the second half.\n[B] Take 200mg of amoxicillin twice daily.\n[A] The match has been played in sunshine conditions.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** Blood pressure reading was 147/75.\n[A] Possession has been 46%-60% so far.\n[B] Limit sodium intake to 1989mg per day.\n\nQuestion 1: Based ONLY on conversation A (about sports commentary), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"31\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_hard_018",
  "task_type": "stream_segregation",
  "difficulty": "Hard",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about sports commentary).\nCompletely ignore conversation B.\n\n[A] The score is 0-0 at the end of the third quarter.\n[B] Blood pressure reading was 146/63.\n[A] Possession has been 49%-62% so far.\n[B] Exercise for at least 12 minutes daily.\n[A] The referee issued a red card for the foul.\n[B] Limit sodium intake to 2420mg per day.\n[A] Freya scored from 16 yards out.\n[B] Limit sodium intake to 2278mg per day.\n[A] The score is 1-0 at the end of the third quarter.\n[B] Blood pressure reading was 126/79.\n[A] The corner kick is taken by Bram.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** Apply the antibiotic cream 3 times per day.\n\nQuestion 1: Based ONLY on conversation A (about sports commentary), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"0\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_hard_019",
  "task_type": "stream_segregation",
  "difficulty": "Hard",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about sports commentary).\nCompletely ignore conversation B.\n\n[A] Magnus scored from 23 yards out.\n[B] Limit sodium intake to 2171mg per day.\n[A] The match has been played in rain conditions.\n[B] Limit sodium intake to 2388mg per day.\n[A] The referee issued a red card for the foul.\n[B] Schedule a follow-up if symptoms persist beyond 3 days.\n[A] The attendance tonight is 33,528 spectators.\n[B] Take 100mg of amoxicillin twice daily.\n[A] The attendance tonight is 19,696 spectators.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** The follow-up appointment is in 8 weeks.\n[A] The match has been played in sunshine conditions.\n[B] Blood pressure reading was 126/91.\n\nQuestion 1: Based ONLY on conversation A (about sports commentary), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"23\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_hard_020",
  "task_type": "stream_segregation",
  "difficulty": "Hard",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversation B.\n\n[A] Harvest when the sunflower reaches 11 inches tall.\n[B] Revenue from the European region grew 25%.\n[A] Add potassium fertilizer once every 2 weeks.\n[B] The quarterly revenue increased by 20% year-over-year.\n[A] Mulch with wood chips to retain moisture.\n[B] Capital expenditure is budgeted at $170 million.\n[A] Prune the sunflower back to 8 inches in October.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** Revenue from the North American region grew 30%.\n[A] Plant the basil seeds 2 inches deep.\n[B] Dividends per share will be $2.72.\n[A] Harvest when the sunflower reaches 14 inches tall.\n[B] Dividends per share will be $2.86.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"11\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_hard_021",
  "task_type": "stream_segregation",
  "difficulty": "Hard",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversation B.\n\n[A] The soil pH should be between 5.6 and 7.2.\n[B] Dividends per share will be $1.71.\n[A] Expect germination in 5 to 19 days.\n[B] Market capitalization reached $116 billion.\n[A] The soil pH should be between 5.9 and 7.3.\n[B] Revenue from the North American region grew 23%.\n[A] Mulch with straw to retain moisture.\n[B] The quarterly revenue increased by 25% year-over-year.\n[A] Space each plant at least 12 inches apart.\n[B] Dividends per share will be $2.21.\n[A] Plant the lettuce seeds 0.25 inches deep.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** The debt-to-equity ratio stands at 1.16.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"5.6\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_hard_022",
  "task_type": "stream_segregation",
  "difficulty": "Hard",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about repair instructions).\nCompletely ignore conversation B.\n\n[A] Let the joint set for at least 23 hours.\n[B] Parking is available for 31 vehicles.\n[A] First, disconnect the power supply completely.\n[B] The cake needs to be ordered 9 days ahead.\n[A] Remove the 3 screws from the back panel.\n[B] Send invitations at least 2 weeks in advance.\n[A] Apply epoxy to both surfaces before joining.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** The photographer charges $249 per hour.\n[A] Locate the thermal fuse — it should be near the gate B.\n[B] Send invitations at least 7 weeks in advance.\n[A] Remove the 3 screws from the back panel.\n[B] The venue holds up to 158 guests.\n\nQuestion 1: Based ONLY on conversation A (about repair instructions), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"23\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_hard_023",
  "task_type": "stream_segregation",
  "difficulty": "Hard",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about cooking recipe).\nCompletely ignore conversation B.\n\n[A] The total cooking time should be about 30 minutes.\n[B] The flight departs at 13:30 from terminal 2.\n[A] Remove from heat and let it cool for 24 minutes.\n[B] Book a hotel near the cathedral for the best location.\n[A] Stir occasionally until the sauce thickens.\n[B] Exchange currency at the airport — the rate is 0.72 to the dollar.\n[A] Dice the zucchini into small cubes.\n[B] Pack an umbrella — the weather forecast shows rain.\n[A] Let the mixture simmer for 33 minutes.\n[B] Exchange currency at the airport — the rate is 0.58 to the dollar.\n[A] Add 60 tablespoons of olive oil to the pan.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** The rental car pickup is at the east exit.\n\nQuestion 1: Based ONLY on conversation A (about cooking recipe), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"30\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_expert_024",
  "task_type": "stream_segregation",
  "difficulty": "Expert",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about sports commentary).\nCompletely ignore conversation B.\n\n[A] The score is 2-4 at the end of the third quarter.\n[B] The follow-up appointment is in 3 weeks.\n[A] The corner kick is taken by Bram.\n[B] The recommended daily water intake is 2.7 liters.\n[A] The corner kick is taken by Magnus.\n[B] Schedule a follow-up if symptoms persist beyond 10 days.\n[A] Possession has been 49%-62% so far.\n[B] Apply the antibiotic cream 2 times per day.\n[A] The referee issued a red card for the foul.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** The recommended daily water intake is 2.0 liters.\n[A] The match has been played in cold winds conditions.\n[B] The follow-up appointment is in 4 weeks.\n[A] Injury time will be 2 minutes.\n[B] Apply the hydrocortisone cream 3 times per day.\n[A] Injury time will be 5 minutes.\n[B] The test results will be available in 12 business days.\n\nQuestion 1: Based ONLY on conversation A (about sports commentary), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"2\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_expert_025",
  "task_type": "stream_segregation",
  "difficulty": "Expert",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversation B.\n\n[A] Prune the sunflower back to 10 inches in October.\n[B] The board approved a $461 million share buyback.\n[A] Plant the basil seeds 2 inches deep.\n[B] The stock trades at a P/E ratio of 23.1.\n[A] Water thoroughly every 10 days during autumn.\n[B] The quarterly revenue increased by 6% year-over-year.\n[A] Space each plant at least 19 inches apart.\n[B] The board approved a $181 million share buyback.\n[A] Harvest when the lettuce reaches 7 inches tall.\n[B] Revenue from the European region grew 7%.\n[A] Water thoroughly every 11 days during autumn.\n[B] The stock trades at a P/E ratio of 28.7.\n[A] Watch for slugs — treat with insecticidal soap if spotted.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** The stock trades at a P/E ratio of 29.3.\n[A] Plant the sunflower seeds 2 inches deep.\n[B] Market capitalization reached $287 billion.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"10\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_expert_026",
  "task_type": "stream_segregation",
  "difficulty": "Expert",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about sports commentary).\nCompletely ignore conversation B.\n\n[A] Injury time will be 5 minutes.\n[B] Apply the antibiotic cream 2 times per day.\n[A] The match has been played in sunshine conditions.\n[B] Blood pressure reading was 130/62.\n[A] The referee issued a yellow card for the foul.\n[B] Schedule a follow-up if symptoms persist beyond 4 days.\n[A] The attendance tonight is 70,976 spectators.\n[B] Schedule a follow-up if symptoms persist beyond 10 days.\n[A] Possession has been 61%-51% so far.\n[B] The test results will be available in 11 business days.\n[A] The corner kick is taken by Zora.\n[B] Limit sodium intake to 1656mg per day.\n[A] Injury time will be 4 minutes.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** Take 100mg of ibuprofen twice daily.\n[A] The corner kick is taken by Greta.\n[B] Apply the hydrocortisone cream 3 times per day.\n\nQuestion 1: Based ONLY on conversation A (about sports commentary), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"5\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_expert_027",
  "task_type": "stream_segregation",
  "difficulty": "Expert",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversation B.\n\n[A] Prune the basil back to 14 inches in October.\n[B] Capital expenditure is budgeted at $88 million.\n[A] Plant the tomato seeds 0.25 inches deep.\n[B] Dividends per share will be $4.46.\n[A] Expect germination in 7 to 20 days.\n[B] Capital expenditure is budgeted at $24 million.\n[A] Plant the basil seeds 1 inches deep.\n[B] Capital expenditure is budgeted at $174 million.\n[A] Water thoroughly every 12 days during spring.\n[B] Revenue from the North American region grew 30%.\n[A] The soil pH should be between 6.2 and 7.4.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** The stock trades at a P/E ratio of 26.3.\n[A] The soil pH should be between 5.5 and 7.3.\n[B] Capital expenditure is budgeted at $154 million.\n[A] Harvest when the lettuce reaches 26 inches tall.\n[B] Dividends per share will be $0.72.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"14\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_expert_028",
  "task_type": "stream_segregation",
  "difficulty": "Expert",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about sports commentary).\nCompletely ignore conversation B.\n\n[A] Freya scored from 18 yards out.\n[B] Avoid caffeine for at least 8 days post-procedure.\n[A] The corner kick is taken by Kenji.\n[B] Take 100mg of ibuprofen twice daily.\n[A] Magnus makes a save from close range.\n[B] Take 250mg of metformin twice daily.\n[A] Possession has been 44%-40% so far.\n[B] The test results will be available in 4 business days.\n[A] The corner kick is taken by Dmitri.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** The recommended daily water intake is 1.8 liters.\n[A] The referee issued a yellow card for the foul.\n[B] Limit sodium intake to 1752mg per day.\n[A] The corner kick is taken by Femi.\n[B] Avoid gluten for at least 6 days post-procedure.\n[A] The score is 2-3 at the end of the first half.\n[B] Schedule a follow-up if symptoms persist beyond 4 days.\n\nQuestion 1: Based ONLY on conversation A (about sports commentary), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"18\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_expert_029",
  "task_type": "stream_segregation",
  "difficulty": "Expert",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about cooking recipe).\nCompletely ignore conversation B.\n\n[A] Remove from heat and let it cool for 22 minutes.\n[B] The flight departs at 8:00 from terminal 1.\n[A] Let the mixture simmer for 21 minutes.\n[B] Check out is at 10:00 — leave bags at reception.\n[A] Serve on a warm plate alongside rice.\n[B] Budget approximately $36 per day for meals.\n[A] The total cooking time should be about 39 minutes.\n[B] Pack sunscreen — the weather forecast shows cold winds.\n[A] Serve on a warm plate alongside salad.\n[B] Budget approximately $95 per day for meals.\n[A] First, preheat the oven to 265 degrees.\n[B] The museum on Zora Street is open until 20:00.\n[A] Remove from heat and let it cool for 22 minutes.\n[B] Book a hotel near the central park for the best location.\n[A] First, preheat the oven to 448 degrees.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** Exchange currency at the airport — the rate is 1.13 to the dollar.\n\nQuestion 1: Based ONLY on conversation A (about cooking recipe), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"22\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_expert_030",
  "task_type": "stream_segregation",
  "difficulty": "Expert",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about cooking recipe).\nCompletely ignore conversation B.\n\n[A] Add 271 tablespoons of olive oil to the pan.\n[B] Book a hotel near the central park for the best location.\n[A] Stir occasionally until the sauce thickens.\n[B] The flight departs at 19:30 from terminal 1.\n[A] First, preheat the oven to 150 degrees.\n[B] The guided tour starts at 14:00 near the main square.\n[A] Remove from heat and let it cool for 33 minutes.\n[B] Budget approximately $60 per day for meals.\n[A] Dice the pepper into small cubes.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** Book a hotel near the central park for the best location.\n[A] Remove from heat and let it cool for 18 minutes.\n[B] The guided tour starts at 9:00 near the main square.\n[A] Remove from heat and let it cool for 13 minutes.\n[B] The rental car pickup is at gate B.\n[A] The total cooking time should be about 54 minutes.\n[B] The guided tour starts at 13:00 near the main square.\n\nQuestion 1: Based ONLY on conversation A (about cooking recipe), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"271\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_expert_031",
  "task_type": "stream_segregation",
  "difficulty": "Expert",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversation B.\n\n[A] Water thoroughly every 3 days during summer.\n[B] The stock trades at a P/E ratio of 18.5.\n[A] Space each plant at least 12 inches apart.\n[B] Operating costs are projected at $172 million.\n[A] Water thoroughly every 14 days during autumn.\n[B] The board approved a $483 million share buyback.\n[A] Water thoroughly every 3 days during summer.\n[B] The quarterly revenue increased by 20% year-over-year.\n[A] Plant the basil seeds 1 inches deep.\n[B] Market capitalization reached $21 billion.\n[A] Water thoroughly every 11 days during spring.\n[B] The debt-to-equity ratio stands at 1.22.\n[A] The soil pH should be between 6.0 and 7.5.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** Dividends per share will be $2.55.\n[A] Plant the sunflower seeds 2 inches deep.\n[B] Revenue from the North American region grew 24%.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"3\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_frontier_032",
  "task_type": "stream_segregation",
  "difficulty": "Frontier",
  "prompt": "Below are THREE interleaved conversations marked [A], [B], and [C].\nAnswer questions ONLY about conversation A (about cooking recipe).\nCompletely ignore conversations B and C.\n\n[A] First, preheat the oven to 392 degrees.\n[B] Pack a warm jacket — the weather forecast shows rain.\n[C] Dividends per share will be $3.65.\n[A] Season with salt, pepper, and a pinch of turmeric.\n[B] Book a hotel near the old market for the best location.\n[C] Revenue from the Asia-Pacific region grew 21%.\n[A] Season with salt, pepper, and a pinch of paprika.\n[B] The flight departs at 14:15 from terminal 3.\n[C] Capital expenditure is budgeted at $162 million.\n[A] Add 421 tablespoons of olive oil to the pan.\n[B] Check out is at 12:00 — leave bags at reception.\n[C] Operating costs are projected at $54 million.\n[A] Dice the zucchini into small cubes.\n[B] Pack an umbrella — the weather forecast shows cold winds.\n[C] Capital expenditure is budgeted at $97 million.\n[A] Dice the celery into small cubes.\n[B] The rental car pickup is at gate B.\n[C] Net profit margin improved to 15.3%.\n[A] First, preheat the oven to 240 degrees.\n[B] The rental car pickup is at the main lobby.\n[C] Net profit margin improved to 19.2%.\n[A] Dice the carrot into small cubes.\n[B] Budget approximately $60 per day for meals.\n[C] Market capitalization reached $280 billion.\n[A] Season with salt, pepper, and a pinch of cumin.\n[B] The museum on Ravi Street is open until 18:00.\n[C] The debt-to-equity ratio stands at 1.16.\n[A] Garnish with fresh basil before serving.\n[B] The flight departs at 17:45 from terminal 1.\n[C] The debt-to-equity ratio stands at 0.47.\n[A] Stir occasionally until the sauce thickens.\n[B] Book a hotel near the central park for the best location.\n[C] The board approved a $239 million share buyback.\n[A] Remove from heat and let it cool for 22 minutes.\n[B] Pack sunscreen — the weather forecast shows sunshine.\n[C] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** Capital expenditure is budgeted at $38 million.\n[A] First, preheat the oven to 378 degrees.\n[B] Book a hotel near the cathedral for the best location.\n[C] Net profit margin improved to 15.4%.\n[A] Dice the pepper into small cubes.\n[B] The train from the airport takes about 38 minutes.\n[C] The stock trades at a P/E ratio of 11.5.\n\nQuestion 1: Based ONLY on conversation A (about cooking recipe), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation C? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"392\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_frontier_033",
  "task_type": "stream_segregation",
  "difficulty": "Frontier",
  "prompt": "Below are THREE interleaved conversations marked [A], [B], and [C].\nAnswer questions ONLY about conversation A (about cooking recipe).\nCompletely ignore conversations B and C.\n\n[A] First, preheat the oven to 447 degrees.\n[B] The rental car pickup is at the east exit.\n[C] Revenue from the North American region grew 21%.\n[A] Add 122 tablespoons of olive oil to the pan.\n[B] Exchange currency at the airport — the rate is 0.87 to the dollar.\n[C] Market capitalization reached $404 billion.\n[A] The total cooking time should be about 81 minutes.\n[B] The guided tour starts at 10:00 near the main square.\n[C] Capital expenditure is budgeted at $51 million.\n[A] Garnish with fresh parsley before serving.\n[B] The flight departs at 9:00 from terminal 2.\n[C] Market capitalization reached $483 billion.\n[A] Let the mixture simmer for 26 minutes.\n[B] The flight departs at 13:45 from terminal 2.\n[C] The quarterly revenue increased by 14% year-over-year.\n[A] The total cooking time should be about 30 minutes.\n[B] The flight departs at 10:00 from terminal 4.\n[C] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** Revenue from the European region grew 12%.\n[A] Let the mixture simmer for 28 minutes.\n[B] Book a hotel near the old market for the best location.\n[C] Dividends per share will be $4.28.\n[A] Let the mixture simmer for 16 minutes.\n[B] The flight departs at 12:00 from terminal 3.\n[C] The quarterly revenue increased by 10% year-over-year.\n[A] Add 433 tablespoons of olive oil to the pan.\n[B] Budget approximately $110 per day for meals.\n[C] Capital expenditure is budgeted at $123 million.\n[A] Stir occasionally until the sauce thickens.\n[B] Exchange currency at the airport — the rate is 1.27 to the dollar.\n[C] Net profit margin improved to 24.6%.\n[A] Add 326 tablespoons of olive oil to the pan.\n[B] Check out is at 11:00 — leave bags at reception.\n[C] The stock trades at a P/E ratio of 15.3.\n[A] Remove from heat and let it cool for 42 minutes.\n[B] Exchange currency at the airport — the rate is 0.57 to the dollar.\n[C] Revenue from the Asia-Pacific region grew 13%.\n[A] First, preheat the oven to 357 degrees.\n[B] The guided tour starts at 11:00 near the main square.\n[C] Revenue from the European region grew 3%.\n[A] Dice the zucchini into small cubes.\n[B] The rental car pickup is at the east exit.\n[C] Market capitalization reached $72 billion.\n\nQuestion 1: Based ONLY on conversation A (about cooking recipe), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation C? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"447\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_frontier_034",
  "task_type": "stream_segregation",
  "difficulty": "Frontier",
  "prompt": "Below are THREE interleaved conversations marked [A], [B], and [C].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversations B and C.\n\n[A] Prune the sunflower back to 7 inches in October.\n[B] The corner kick is taken by Ravi.\n[C] The rental car pickup is at the main lobby.\n[A] Space each plant at least 9 inches apart.\n[B] The referee issued a red card for the foul.\n[C] Budget approximately $81 per day for meals.\n[A] Harvest when the tomato reaches 24 inches tall.\n[B] The referee issued a red card for the foul.\n[C] The rental car pickup is at the main lobby.\n[A] Mulch with leaf compost to retain moisture.\n[B] The score is 1-4 at the end of the third quarter.\n[C] The flight departs at 9:00 from terminal 1.\n[A] Add potassium fertilizer once every 3 weeks.\n[B] The corner kick is taken by Zain.\n[C] The guided tour starts at 14:00 near the main square.\n[A] Water thoroughly every 10 days during spring.\n[B] Injury time will be 5 minutes.\n[C] Pack an umbrella — the weather forecast shows cold winds.\n[A] The soil pH should be between 6.3 and 7.1.\n[B] Substitution: Idris replaces Joelle.\n[C] Pack sunscreen — the weather forecast shows cold winds.\n[A] Prune the lettuce back to 12 inches in April.\n[B] Injury time will be 3 minutes.\n[C] Check out is at 12:00 — leave bags at reception.\n[A] Prune the lettuce back to 17 inches in October.\n[B] Substitution: Nico replaces Kenji.\n[C] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** The flight departs at 9:30 from terminal 2.\n[A] Add potassium fertilizer once every 5 weeks.\n[B] The corner kick is taken by Orla.\n[C] Book a hotel near the central park for the best location.\n[A] Mulch with wood chips to retain moisture.\n[B] The referee issued a red card for the foul.\n[C] Check out is at 12:00 — leave bags at reception.\n[A] The soil pH should be between 6.2 and 7.4.\n[B] Zain scored from 32 yards out.\n[C] Budget approximately $101 per day for meals.\n[A] Space each plant at least 18 inches apart.\n[B] Sigrid makes a save from close range.\n[C] Budget approximately $55 per day for meals.\n[A] Add potassium fertilizer once every 2 weeks.\n[B] The referee issued a yellow card for the foul.\n[C] Budget approximately $139 per day for meals.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation C? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"7\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_frontier_035",
  "task_type": "stream_segregation",
  "difficulty": "Frontier",
  "prompt": "Below are THREE interleaved conversations marked [A], [B], and [C].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversations B and C.\n\n[A] Add nitrogen-rich fertilizer once every 2 weeks.\n[B] The score is 2-0 at the end of the third quarter.\n[C] The guided tour starts at 11:00 near the main square.\n[A] Water thoroughly every 2 days during spring.\n[B] The match has been played in sunshine conditions.\n[C] Book a hotel near the cathedral for the best location.\n[A] Add potassium fertilizer once every 7 weeks.\n[B] The match has been played in cold winds conditions.\n[C] Budget approximately $149 per day for meals.\n[A] Plant the basil seeds 2 inches deep.\n[B] The match has been played in cold winds conditions.\n[C] Book a hotel near the central park for the best location.\n[A] Watch for slugs — treat with neem oil if spotted.\n[B] Possession has been 61%-51% so far.\n[C] Exchange currency at the airport — the rate is 0.97 to the dollar.\n[A] Harvest when the tomato reaches 12 inches tall.\n[B] Possession has been 43%-44% so far.\n[C] The flight departs at 13:15 from terminal 3.\n[A] Space each plant at least 22 inches apart.\n[B] Colette scored from 10 yards out.\n[C] Book a hotel near the central park for the best location.\n[A] The soil pH should be between 6.3 and 7.1.\n[B] The match has been played in sunshine conditions.\n[C] The flight departs at 12:45 from terminal 3.\n[A] Space each plant at least 6 inches apart.\n[B] Olena makes a save from close range.\n[C] The museum on Viktor Street is open until 19:00.\n[A] Water thoroughly every 8 days during autumn.\n[B] The score is 0-4 at the end of the second half.\n[C] The guided tour starts at 12:00 near the main square.\n[A] The soil pH should be between 5.7 and 6.9.\n[B] The referee issued a yellow card for the foul.\n[C] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** The rental car pickup is at gate B.\n[A] Harvest when the sunflower reaches 22 inches tall.\n[B] Qadir makes a save from close range.\n[C] The museum on Femi Street is open until 17:00.\n[A] Mulch with straw to retain moisture.\n[B] Possession has been 43%-46% so far.\n[C] The train from the airport takes about 6 minutes.\n[A] Harvest when the tomato reaches 33 inches tall.\n[B] The attendance tonight is 28,263 spectators.\n[C] Pack an umbrella — the weather forecast shows rain.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation C? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"2\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_frontier_036",
  "task_type": "stream_segregation",
  "difficulty": "Frontier",
  "prompt": "Below are THREE interleaved conversations marked [A], [B], and [C].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversations B and C.\n\n[A] Expect germination in 10 to 12 days.\n[B] The referee issued a yellow card for the foul.\n[C] The guided tour starts at 10:00 near the main square.\n[A] Expect germination in 7 to 12 days.\n[B] The attendance tonight is 31,466 spectators.\n[C] The museum on Lumi Street is open until 20:00.\n[A] The soil pH should be between 6.1 and 7.4.\n[B] Injury time will be 2 minutes.\n[C] Check out is at 12:00 — leave bags at reception.\n[A] Expect germination in 10 to 16 days.\n[B] The attendance tonight is 22,742 spectators.\n[C] Budget approximately $127 per day for meals.\n[A] Watch for slugs — treat with diatomaceous earth if spotted.\n[B] Yuki scored from 9 yards out.\n[C] The train from the airport takes about 16 minutes.\n[A] Plant the lettuce seeds 1 inches deep.\n[B] Femi makes a save from close range.\n[C] The train from the airport takes about 39 minutes.\n[A] Plant the tomato seeds 1 inches deep.\n[B] The corner kick is taken by Hana.\n[C] The rental car pickup is at gate B.\n[A] Prune the tomato back to 10 inches in March.\n[B] The score is 4-4 at the end of the first half.\n[C] Book a hotel near the cathedral for the best location.\n[A] Prune the lettuce back to 17 inches in April.\n[B] The corner kick is taken by Nalini.\n[C] The guided tour starts at 10:00 near the main square.\n[A] The soil pH should be between 5.7 and 7.4.\n[B] The attendance tonight is 25,945 spectators.\n[C] The museum on Kenji Street is open until 20:00.\n[A] Plant the basil seeds 2 inches deep.\n[B] Injury time will be 5 minutes.\n[C] Book a hotel near the cathedral for the best location.\n[A] Watch for caterpillars — treat with insecticidal soap if spotted.\n[B] The match has been played in rain conditions.\n[C] The flight departs at 8:00 from terminal 1.\n[A] Mulch with straw to retain moisture.\n[B] The match has been played in rain conditions.\n[C] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** The train from the airport takes about 37 minutes.\n[A] Harvest when the tomato reaches 29 inches tall.\n[B] The score is 2-4 at the end of the second half.\n[C] The train from the airport takes about 5 minutes.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation C? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"10\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_frontier_037",
  "task_type": "stream_segregation",
  "difficulty": "Frontier",
  "prompt": "Below are THREE interleaved conversations marked [A], [B], and [C].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversations B and C.\n\n[A] Space each plant at least 19 inches apart.\n[B] The corner kick is taken by Joelle.\n[C] The museum on Freya Street is open until 19:00.\n[A] Space each plant at least 12 inches apart.\n[B] Injury time will be 4 minutes.\n[C] The rental car pickup is at the east exit.\n[A] Expect germination in 7 to 18 days.\n[B] The attendance tonight is 72,168 spectators.\n[C] The museum on Dariush Street is open until 21:00.\n[A] Water thoroughly every 8 days during spring.\n[B] Nalini scored from 18 yards out.\n[C] Check out is at 12:00 — leave bags at reception.\n[A] Prune the lettuce back to 15 inches in March.\n[B] The match has been played in cold winds conditions.\n[C] The train from the airport takes about 42 minutes.\n[A] Plant the basil seeds 0.5 inches deep.\n[B] The score is 4-4 at the end of the second half.\n[C] The rental car pickup is at the main lobby.\n[A] Prune the lettuce back to 6 inches in October.\n[B] The match has been played in cold winds conditions.\n[C] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** The guided tour starts at 13:00 near the main square.\n[A] Space each plant at least 7 inches apart.\n[B] Injury time will be 2 minutes.\n[C] The rental car pickup is at the main lobby.\n[A] Expect germination in 6 to 21 days.\n[B] Paloma scored from 35 yards out.\n[C] Exchange currency at the airport — the rate is 1.41 to the dollar.\n[A] Water thoroughly every 2 days during autumn.\n[B] The attendance tonight is 46,816 spectators.\n[C] The flight departs at 14:00 from terminal 4.\n[A] Plant the tomato seeds 1 inches deep.\n[B] The attendance tonight is 29,416 spectators.\n[C] The museum on Haruto Street is open until 20:00.\n[A] Expect germination in 7 to 13 days.\n[B] The match has been played in sunshine conditions.\n[C] The guided tour starts at 9:00 near the main square.\n[A] Plant the lettuce seeds 2 inches deep.\n[B] Possession has been 53%-57% so far.\n[C] Exchange currency at the airport — the rate is 0.65 to the dollar.\n[A] Watch for slugs — treat with insecticidal soap if spotted.\n[B] Injury time will be 4 minutes.\n[C] Pack a warm jacket — the weather forecast shows cold winds.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation C? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"19\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_frontier_038",
  "task_type": "stream_segregation",
  "difficulty": "Frontier",
  "prompt": "Below are THREE interleaved conversations marked [A], [B], and [C].\nAnswer questions ONLY about conversation A (about cooking recipe).\nCompletely ignore conversations B and C.\n\n[A] Remove from heat and let it cool for 9 minutes.\n[B] If the issue persists, check the wiring harness.\n[C] The test results will be available in 11 business days.\n[A] The total cooking time should be about 33 minutes.\n[B] Test the operation before restoring power.\n[C] The recommended daily water intake is 1.9 liters.\n[A] Let the mixture simmer for 38 minutes.\n[B] Remove the 2 screws from the back panel.\n[C] The recommended daily water intake is 2.5 liters.\n[A] First, preheat the oven to 305 degrees.\n[B] Replace the worn gasket with the new one from the kit.\n[C] Blood pressure reading was 145/66.\n[A] Add 441 tablespoons of olive oil to the pan.\n[B] Let the joint set for at least 4 hours.\n[C] The follow-up appointment is in 7 weeks.\n[A] Let the mixture simmer for 40 minutes.\n[B] First, disconnect the power supply completely.\n[C] Take 250mg of metformin twice daily.\n[A] Serve on a warm plate alongside potatoes.\n[B] Locate the relay switch — it should be near the gate B.\n[C] Exercise for at least 30 minutes daily.\n[A] Dice the celery into small cubes.\n[B] Remove the 2 screws from the back panel.\n[C] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** The recommended daily water intake is 1.9 liters.\n[A] Let the mixture simmer for 40 minutes.\n[B] Use a 8mm wrench to loosen the bolt.\n[C] Avoid gluten for at least 8 days post-procedure.\n[A] Season with salt, pepper, and a pinch of cumin.\n[B] Locate the thermal fuse — it should be near the gate B.\n[C] Avoid alcohol for at least 4 days post-procedure.\n[A] Season with salt, pepper, and a pinch of cumin.\n[B] Reattach the panel and tighten screws to 24 Nm.\n[C] The follow-up appointment is in 2 weeks.\n[A] Stir occasionally until the sauce thickens.\n[B] Reattach the panel and tighten screws to 14 Nm.\n[C] The test results will be available in 5 business days.\n[A] Dice the zucchini into small cubes.\n[B] Locate the thermal fuse — it should be near the the main lobby.\n[C] The recommended daily water intake is 1.7 liters.\n[A] Season with salt, pepper, and a pinch of paprika.\n[B] Reattach the panel and tighten screws to 24 Nm.\n[C] The follow-up appointment is in 6 weeks.\n\nQuestion 1: Based ONLY on conversation A (about cooking recipe), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation C? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"9\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_frontier_039",
  "task_type": "stream_segregation",
  "difficulty": "Frontier",
  "prompt": "Below are THREE interleaved conversations marked [A], [B], and [C].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversations B and C.\n\n[A] Plant the lettuce seeds 2 inches deep.\n[B] Injury time will be 2 minutes.\n[C] Check out is at 12:00 — leave bags at reception.\n[A] Harvest when the lettuce reaches 35 inches tall.\n[B] Possession has been 36%-50% so far.\n[C] Book a hotel near the old market for the best location.\n[A] Prune the basil back to 13 inches in October.\n[B] The match has been played in rain conditions.\n[C] Budget approximately $88 per day for meals.\n[A] Water thoroughly every 13 days during summer.\n[B] Injury time will be 2 minutes.\n[C] The museum on Xander Street is open until 21:00.\n[A] Watch for aphids — treat with insecticidal soap if spotted.\n[B] Sigrid makes a save from close range.\n[C] The museum on Orla Street is open until 21:00.\n[A] Space each plant at least 20 inches apart.\n[B] Substitution: Viktor replaces Viktor.\n[C] The museum on Kenji Street is open until 18:00.\n[A] Plant the basil seeds 0.5 inches deep.\n[B] The score is 1-3 at the end of the third quarter.\n[C] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE — THIS IS AN ALERT — immediate attention required. ALERT ***** The flight departs at 20:30 from terminal 5.\n[A] Prune the lettuce back to 11 inches in October.\n[B] The match has been played in cold winds conditions.\n[C] The train from the airport takes about 10 minutes.\n[A] Add nitrogen-rich fertilizer once every 8 weeks.\n[B] Injury time will be 2 minutes.\n[C] Budget approximately $127 per day for meals.\n[A] Water thoroughly every 6 days during autumn.\n[B] Injury time will be 2 minutes.\n[C] Budget approximately $102 per day for meals.\n[A] Water thoroughly every 9 days during summer.\n[B] The score is 2-3 at the end of the third quarter.\n[C] The guided tour starts at 14:00 near the main square.\n[A] Expect germination in 7 to 12 days.\n[B] The score is 2-2 at the end of the second half.\n[C] Pack a warm jacket — the weather forecast shows rain.\n[A] Harvest when the tomato reaches 6 inches tall.\n[B] Injury time will be 4 minutes.\n[C] The train from the airport takes about 32 minutes.\n[A] Space each plant at least 24 inches apart.\n[B] The corner kick is taken by Tariq.\n[C] Pack a warm jacket — the weather forecast shows cold winds.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation C? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"2\", \"has_breakthrough\": true}"
 }
]
''')

print(f"Loaded {len(DATASET)} items")
for tt in ['stream_segregation']:
    count = sum(1 for d in DATASET if d["task_type"] == tt)
    print(f"  {tt}: {count} items")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 4: Execution Loop
# ══════════════════════════════════════════════════════════════════════

TASK_DISPATCH = {
    "stream_segregation": cogattention_stream_segregation,
}

n_total = len(DATASET)
for i, item in enumerate(DATASET):
    task_fn = TASK_DISPATCH[item["task_type"]]
    print(f"[{i+1}/{n_total}] {item['task_id']} ({item['difficulty']})")
    task_fn.run(
        llm=kbench.llm,
        prompt=item["prompt"],
        gold_json=item["gold_json"],
        task_id=item["task_id"],
        difficulty=item["difficulty"],
    )

print(f"\nCompleted {n_total} items for Sustained Attention")
